In [42]:
# Optional: Install required dependencies if not already present in your Kaggle environment
!pip install -q tensorly datasets scikit-learn


time: 3.69s
cummulative_time: 29.39s
time: 3.69s
cummulative_time: 29.39s
time: 3.69s
cummulative_time: 29.39s
time: 3.69s
cummulative_time: 29.39s
time: 3.69s
cummulative_time: 29.39s
time: 3.69s
cummulative_time: 29.39s
time: 3.69s
cummulative_time: 29.39s


# Experiment 14: Model-Wide Inactive Weights DBSCAN 3D Tensor Compression (Kaggle Version)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Key Architectural Highlights:
1. **Fully Self-Contained**:
   - Zero dependencies on custom modules (`neural_decomp` has been completely inlined into native Hugging Face `transformers` and PyTorch).
   - Designed to run out-of-the-box in Kaggle (NVIDIA T4 / P100 / A100 GPU environments).
2. **Notebook 02 Inactive Preprocessing Pipeline**:
   - **Low-Rank Spectral Denoising (SVD 95% Energy)**: Eliminates noisy singular dimensions from the inactive subspace of each of the 78 projection matrices.
   - **Magnitude Sparsification (60th Percentile)**: Zero-masks near-zero residual noise, acting as an implicit regularizer.
3. **RAM-Optimized Streaming Hook Profiling**:
   - Computes activation summaries along the token dimension immediately inside hooks to maintain a light memory footprint (<350 MB RAM total), preventing system crashes and OOM errors.
4. **Inactive 3D Tensors**:
   - Groups 10 uniform chunks of size 400 into 3D tensors: $\mathcal{T}_{\text{inact}} \in \mathbb{R}^{10 \times 400 \times 1152}$ ($4,000$ inactive coordinates factorized per matrix, $312,000$ coordinates model-wide).
5. **Multi-Tier Model-Wide Compression**:
   - **Tier 1: Inactive Moderate** (`[5, 120, 350]`): Eliminates **~304 Million parameters**.
   - **Tier 2: Inactive Aggressive** (`[4, 80, 200]`): Eliminates **~334 Million parameters**.
   - **Tier 3: Inactive Ultra-Aggressive** (`[3, 40, 100]`): Eliminates **~350 Million parameters**.

In [43]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

time: 0.10s
cummulative_time: 29.49s
time: 0.10s
cummulative_time: 29.50s
time: 0.10s
cummulative_time: 29.50s
time: 0.10s
cummulative_time: 29.50s
time: 0.10s
cummulative_time: 29.50s
time: 0.10s
cummulative_time: 29.50s
time: 0.10s
cummulative_time: 29.50s


In [44]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Standard Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

print("Environment configured.")
print("TensorLy Backend:", tl.get_backend())
print("PyTorch Version: ", torch.__version__)
print("CUDA Available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:     ", torch.cuda.get_device_name(0))


Environment configured.
TensorLy Backend: pytorch
PyTorch Version:  2.10.0+cu128
CUDA Available:   True
Device Name:      Tesla T4


In [45]:
# =====================================================================
# STEP 2: Model & Dataset Loading (Pure Hugging Face - No Custom Module)
# =====================================================================
import huggingface_hub

MODEL_ID = "google/gemma-3-1b-it"
NUM_LAYERS = 26
NUM_EVAL_SAMPLES = 150

# Hugging Face Authentication for Gated Gemma-3 Model
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Authenticated with Hugging Face via token.")
else:
    print("No HF_TOKEN found in Kaggle Secrets or environment.")
    print("If you haven't yet, get your read token from https://huggingface.co/settings/tokens")
    print("and accept terms at https://huggingface.co/google/gemma-3-1b-it")
    huggingface_hub.login()

print(f"Loading model: {MODEL_ID} with device_map='auto' in FP32...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto",
    token=secret_value_0,
)

# Cache all 26 layers' pristine weights on CPU
W_orig_all = {}
for l in range(NUM_LAYERS):
    layer_mlp = model.model.layers[l].mlp
    W_orig_all[l] = {
        "gate_proj": layer_mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   layer_mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": layer_mlp.down_proj.weight.data.clone().cpu(),
    }

print(f"Cached all {NUM_LAYERS} layers pristine weights on CPU (78 projection matrices).")

# Load GLUE MNLI dataset
print(f"\nLoading GLUE MNLI dataset ({NUM_EVAL_SAMPLES} evaluation samples)...")
dataset = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = dataset.select(range(NUM_EVAL_SAMPLES))

label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
label_tokens = ["entailment", "neutral", "contradiction"]
# Note: Gemma SentencePiece tokenization requires leading space for model answer generation
label_token_ids = [tokenizer.encode(" " + tok, add_special_tokens=False)[0] for tok in label_tokens]
print(f"Candidate label token IDs: {list(zip(label_tokens, label_token_ids))}")


Authenticated with Hugging Face via token.
Loading model: google/gemma-3-1b-it with device_map='auto' in FP32...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Cached all 26 layers pristine weights on CPU (78 projection matrices).

Loading GLUE MNLI dataset (150 evaluation samples)...
Candidate label token IDs: [('entailment', 83155), ('neutral', 12643), ('contradiction', 38912)]
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s
time: 9.19s
cummulative_time: 9.20s


In [46]:
# =====================================================================
# STEP 3: Model-Wide Tri-Hook Profiling & Pristine Baseline Accuracy
# =====================================================================
# Memory-optimized streaming storage: pool sequence tokens immediately to prevent RAM bloat
layer_acts = {l: {"gate_proj": [], "up_proj": [], "down_proj": []} for l in range(NUM_LAYERS)}
hooks = []

for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp

    def make_gate_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["gate_proj"].append(
            out.detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    def make_up_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["up_proj"].append(
            out.detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    def make_down_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["down_proj"].append(
            inp[0].detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    hooks.append(lmod.act_fn.register_forward_hook(make_gate_hook(l)))
    hooks.append(lmod.up_proj.register_forward_hook(make_up_hook(l)))
    hooks.append(lmod.down_proj.register_forward_hook(make_down_hook(l)))

baseline_preds, ground_truths = [], []
model.eval()
print(f"Running baseline profiling pass across all {NUM_LAYERS} layers ({NUM_EVAL_SAMPLES} samples)...")
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        baseline_preds.append(pred_label)
        ground_truths.append(sample["label"])

for h in hooks:
    h.remove()

baseline_accuracy = accuracy_score(ground_truths, baseline_preds)
print(f"\nPristine Baseline Accuracy across all 26 layers: {baseline_accuracy * 100:.2f}%")

# Aggregate activation matrices per layer & submodule (Shape: [150, 6912])
acts_matrix_all = {l: {} for l in range(NUM_LAYERS)}
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        acts_matrix_all[l][sub_name] = torch.stack(layer_acts[l][sub_name], dim=0).numpy()

print(f"Successfully captured activation statistics for all 78 submodules ({acts_matrix_all[0]['gate_proj'].shape}).")


Running baseline profiling pass across all 26 layers (150 samples)...


Baseline Profiling: 100%|██████████| 150/150 [00:19<00:00,  7.68it/s]



Pristine Baseline Accuracy across all 26 layers: 50.67%
Successfully captured activation statistics for all 78 submodules ((150, 6912)).
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s
time: 19.72s
cummulative_time: 28.92s


In [47]:
# =====================================================================
# STEP 4: Notebook 02 Denoising, Thresholding & Inactive DBSCAN Clustering
# =====================================================================
NUM_ACTIVE = 2400
INACT_CHUNK_SIZE = 400
INACT_NUM_CHUNKS = 10  # 4,000 inactive coords factorized per matrix

layer_inactive_data = {}

def cluster_inactive_submodule(acts_matrix, weight_tensor, is_col=False, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
    v_all = np.mean(np.abs(acts_matrix), axis=0)
    var_all = np.var(acts_matrix, axis=0)
    
    sorted_all = np.argsort(v_all)
    inactive_pool = sorted_all[:-NUM_ACTIVE]
    active_pool = sorted_all[-NUM_ACTIVE:]
    
    # 1. Notebook 02 Inactive Preprocessing: SVD 95% Denoising + 60% Sparsification Threshold on GPU
    W_sub_inact = (weight_tensor[:, inactive_pool].T if is_col else weight_tensor[inactive_pool, :]).to(device)
    U, S, Vh = torch.linalg.svd(W_sub_inact, full_matrices=False)
    cum_e = torch.cumsum(S**2, dim=0) / torch.sum(S**2)
    r95 = (cum_e >= 0.95).nonzero()[0].item() + 1
    W_denoised = U[:, :r95] @ torch.diag(S[:r95]) @ Vh[:r95, :]
    eps_val = torch.quantile(torch.abs(W_denoised), 0.60)
    W_sparse = W_denoised.clone()
    W_sparse[torch.abs(W_sparse) < eps_val] = 0.0
    
    W_clean = weight_tensor.clone().to(device)
    if is_col:
        W_clean[:, inactive_pool] = W_sparse.T
    else:
        W_clean[inactive_pool, :] = W_sparse
    W_clean = W_clean.cpu()
    
    # 2. Fine-precision DBSCAN on inactive activation values
    v_inact = v_all[inactive_pool]
    eps_inact = max(1e-4, float(np.std(v_inact) * 0.18))
    
    db = DBSCAN(eps=eps_inact, min_samples=30, metric="euclidean")
    labels = db.fit_predict(v_inact.reshape(-1, 1))
    
    # 3. Superweight & Outlier Quarantine
    inact_mags = np.max(np.abs(acts_matrix[:, inactive_pool]), axis=0)
    inact_vars = var_all[inactive_pool]
    super_mask = (labels == -1) | (inact_mags >= np.quantile(inact_mags, 0.99)) | (inact_vars >= np.quantile(inact_vars, 0.99))
    
    super_coords = inactive_pool[super_mask]
    clustered_coords = inactive_pool[~super_mask]
    
    # 4. Form 10 chunks of size 400
    chunk_list = []
    unique_labs = [l for l in np.unique(labels) if l != -1]
    for lab in unique_labs:
        c_sub_idx = np.where((labels == lab) & (~super_mask))[0]
        if len(c_sub_idx) == 0:
            continue
        c_coords = inactive_pool[c_sub_idx]
        sorted_c = c_coords[np.argsort(v_all[c_coords])]
        num_full = len(sorted_c) // INACT_CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_c[ci * INACT_CHUNK_SIZE : (ci + 1) * INACT_CHUNK_SIZE])
            if len(chunk_list) >= INACT_NUM_CHUNKS:
                break
        if len(chunk_list) >= INACT_NUM_CHUNKS:
            break
            
    if len(chunk_list) < INACT_NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [c for c in clustered_coords if c not in assigned]
        needed = INACT_NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= INACT_CHUNK_SIZE:
                chunk_list.append(np.array(avail[:INACT_CHUNK_SIZE]))
                avail = avail[INACT_CHUNK_SIZE:]
                
    # 5. Assemble 3D Inactive Tensor from Denoised & Thresholded Weights
    if is_col:
        T_inact = torch.stack([W_clean[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        T_inact = torch.stack([W_clean[c, :].float().cpu() for c in chunk_list], dim=0)
        
    return {
        "tensor": T_inact,
        "chunk_list": chunk_list,
        "super_coords": super_coords,
        "active_pool": active_pool,
        "is_col": is_col,
    }

print("Running SVD denoising, thresholding, and inactive DBSCAN clustering across all 26 layers on GPU...")
for l in tqdm(range(NUM_LAYERS), desc="Pre-Clustering Inactive"):
    layer_inactive_data[l] = {}
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        is_col = (sub_name == "down_proj")
        sub_mod = getattr(model.model.layers[l].mlp, sub_name)
        layer_inactive_data[l][sub_name] = cluster_inactive_submodule(
            acts_matrix_all[l][sub_name],
            W_orig_all[l][sub_name],
            is_col=is_col,
            device=sub_mod.weight.device,
        )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Completed inactive clustering for all 78 submodules on GPU.")


Running SVD denoising, thresholding, and inactive DBSCAN clustering across all 26 layers on GPU...


Pre-Clustering Inactive: 100%|██████████| 26/26 [00:29<00:00,  1.13s/it]

Completed inactive clustering for all 78 submodules on GPU.
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s
time: 29.50s
cummulative_time: 58.44s


In [48]:
# =====================================================================
# STEP 5: Define Tucker GD Optimizer & Inactive Evaluation Tiers
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    safe_ranks = [
        min(ranks[0], T.shape[0]),
        min(ranks[1], T.shape[1]),
        min(ranks[2], T.shape[2]),
    ]
    T_target = T.to(device)
    core_init, factors_init = tucker(T_target, rank=safe_ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone())
    factors_param = [torch.nn.Parameter(f.clone()) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param))
        final_err = (torch.norm(T_target - T_recon_final) / torch.norm(T_target)).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

inactive_tiers = [
    {
        "name": "Inactive Conservative Tier ([6, 160, 500])",
        "ranks": [6, 160, 500],
    },
    {
        "name": "Inactive Moderate Tier ([5, 120, 350])",
        "ranks": [5, 120, 350],
    },
    {
        "name": "Inactive Aggressive Tier ([4, 80, 200])",
        "ranks": [4, 80, 200],
    },
]

print("Defined GPU Tucker GD optimizer and 3 model-wide inactive tiers.")


Defined GPU Tucker GD optimizer and 3 model-wide inactive tiers.
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s
time: 0.00s
cummulative_time: 58.45s


In [49]:
# =====================================================================
# STEP 6: Execute Model-Wide Inactive Sweeps & GLUE MNLI Evaluation
# =====================================================================
tier_benchmarks = []

for tier in inactive_tiers:
    tier_name = tier["name"]
    ranks = tier["ranks"]
    
    print(f"\n{'='*95}")
    print(f"Running Full-Model Inactive Evaluation: {tier_name}")
    print(f"{'='*95}")
    
    tier_gate_errs, tier_up_errs, tier_down_errs = [], [], []
    total_params_saved = 0
    
    # 1. Factorize inactive tensors and inject across all 26 layers
    for l in range(NUM_LAYERS):
        lmod = model.model.layers[l].mlp
        sdata_layer = layer_inactive_data[l]
        
        for sub_name in ["gate_proj", "up_proj", "down_proj"]:
            sdata = sdata_layer[sub_name]
            T_inact = sdata["tensor"]
            mod_ref = getattr(lmod, sub_name)
            sub_dev = mod_ref.weight.device
            
            cg, fg, T_recon, err = optimize_tucker_gd(
                T_inact, ranks=ranks, num_steps=35, lr=1e-3, device=sub_dev
            )
            
            if sub_name == "gate_proj": tier_gate_errs.append(err)
            elif sub_name == "up_proj":  tier_up_errs.append(err)
            elif sub_name == "down_proj": tier_down_errs.append(err)
            
            orig_p = T_inact.numel()
            comp_p = cg.numel() + sum(f.numel() for f in fg)
            total_params_saved += (orig_p - comp_p)
            
            # Live injection on GPU
            orig_w = W_orig_all[l][sub_name]
            mod_ref.weight.data = orig_w.clone().to(sub_dev)
            
            if sdata["is_col"]:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[:, c] = T_recon[k].T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
                if len(sdata["super_coords"]) > 0:
                    mod_ref.weight.data[:, sdata["super_coords"]] = orig_w[:, sdata["super_coords"]].to(sub_dev)
            else:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[c, :] = T_recon[k].to(device=sub_dev, dtype=mod_ref.weight.dtype)
                if len(sdata["super_coords"]) > 0:
                    mod_ref.weight.data[sdata["super_coords"], :] = orig_w[sdata["super_coords"], :].to(sub_dev)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_gate = np.mean(tier_gate_errs) * 100
    mean_up   = np.mean(tier_up_errs) * 100
    mean_down = np.mean(tier_down_errs) * 100
    
    print(f"Layer Factorization Complete across 78 submodules.")
    print(f"  Mean Inactive Recon Errors: gate={mean_gate:.1f}%, up={mean_up:.1f}%, down={mean_down:.1f}%")
    print(f"  Total Inactive Parameters Eliminated: {total_params_saved:,}")

    # 2. Evaluate downstream on GLUE MNLI
    preds, gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            preds.append(pred_label)
            gts.append(sample["label"])

    acc = accuracy_score(gts, preds)
    delta = acc - baseline_accuracy

    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Inactive Params Cut: {total_params_saved:,}")

    tier_benchmarks.append({
        "Variant": tier_name,
        "Ranks": ranks,
        "Mean_Gate_Err": round(mean_gate, 2),
        "Mean_Up_Err": round(mean_up, 2),
        "Mean_Down_Err": round(mean_down, 2),
        "Params_Eliminated": total_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore pristine model weights across all 26 layers
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sub_mod = getattr(model.model.layers[l].mlp, sub_name)
        sub_mod.weight.data = W_orig_all[l][sub_name].clone().to(sub_mod.weight.device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRestored all 26 layers to pristine weights.")



Running Full-Model Inactive Evaluation: Inactive Conservative Tier ([6, 160, 500])
Layer Factorization Complete across 78 submodules.
  Mean Inactive Recon Errors: gate=81.7%, up=84.1%, down=84.1%
  Total Inactive Parameters Eliminated: 272,059,320


Evaluating Inactive Conservative Tier ([6, 160, 500]): 100%|██████████| 150/150 [00:14<00:00, 10.59it/s]



Result for Inactive Conservative Tier ([6, 160, 500]):
  Downstream Accuracy: 41.33% (Δ vs Baseline: -9.33%)
  Inactive Params Cut: 272,059,320

Running Full-Model Inactive Evaluation: Inactive Moderate Tier ([5, 120, 350])
Layer Factorization Complete across 78 submodules.
  Mean Inactive Recon Errors: gate=87.7%, up=89.8%, down=89.7%
  Total Inactive Parameters Eliminated: 307,846,500


Evaluating Inactive Moderate Tier ([5, 120, 350]): 100%|██████████| 150/150 [00:13<00:00, 10.80it/s]



Result for Inactive Moderate Tier ([5, 120, 350]):
  Downstream Accuracy: 41.33% (Δ vs Baseline: -9.33%)
  Inactive Params Cut: 307,846,500

Running Full-Model Inactive Evaluation: Inactive Aggressive Tier ([4, 80, 200])
Layer Factorization Complete across 78 submodules.
  Mean Inactive Recon Errors: gate=92.6%, up=94.4%, down=94.2%
  Total Inactive Parameters Eliminated: 333,961,680


Evaluating Inactive Aggressive Tier ([4, 80, 200]): 100%|██████████| 150/150 [00:14<00:00, 10.66it/s]



Result for Inactive Aggressive Tier ([4, 80, 200]):
  Downstream Accuracy: 38.67% (Δ vs Baseline: -12.00%)
  Inactive Params Cut: 333,961,680

Restored all 26 layers to pristine weights.
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s
time: 311.60s
cummulative_time: 370.06s


In [50]:
# =====================================================================
# STEP 7: Benchmark Summary & JSON Artifact Export
# =====================================================================
print(f"\n{'='*115}")
print(f"{'Variant':<42} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<12} | {'Accuracy':<9} | {'Delta':<8}")
print(f"{'='*115}")
print(f"{'Baseline (Uncompressed)':<42} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<12} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for b in tier_benchmarks:
    print(f"{b['Variant']:<42} | {b['Mean_Gate_Err']:>6.2f}%  | {b['Mean_Up_Err']:>5.2f}%  | {b['Mean_Down_Err']:>6.2f}%   | {b['Params_Eliminated']:<12,d} | {b['Accuracy']:>7.2f}% | {b['Delta']:>+6.2f}%")
print(f"{'='*115}")

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "14_all_layers_inactive_results.json"

payload = {
    "model_id": MODEL_ID,
    "num_layers": NUM_LAYERS,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "benchmarks": tier_benchmarks,
}

with open(results_file, "w") as f:
    json.dump(payload, f, indent=2)

print(f"\nSaved benchmark results to {results_file}")



Variant                                    | Gate Err  | Up Err   | Down Err  | Params Cut   | Accuracy  | Delta   
Baseline (Uncompressed)                    | 0.00%     | 0.00%    | 0.00%     | 0            |   50.67% | +0.00%  
Inactive Conservative Tier ([6, 160, 500]) |  81.66%  | 84.09%  |  84.06%   | 272,059,320  |   41.33% |  -9.33%
Inactive Moderate Tier ([5, 120, 350])     |  87.66%  | 89.83%  |  89.74%   | 307,846,500  |   41.33% |  -9.33%
Inactive Aggressive Tier ([4, 80, 200])    |  92.59%  | 94.37%  |  94.22%   | 333,961,680  |   38.67% | -12.00%

Saved benchmark results to artifacts/14_all_layers_inactive_results.json
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
time: 0.00s
cummulative_time: 370.07s
